# Test the Retrained Face Recognition Model

In this notebook we test the ONNX model exported in notebook 03 on **still images** and **video**.

The retrained model has learned to distinguish *your* face from other faces:
- **Green** bounding boxes → your face (the class the model was trained on)
- **Red** bounding boxes → other / unknown faces

We'll run inference locally using the Ultralytics YOLO runtime with the ONNX backend.
At the end, we upload the trained model to MinIO so the model server serves it in notebook 04.

In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

from ultralytics import YOLO
import numpy as np
import cv2
from matplotlib import pyplot as plt
from pathlib import Path
from IPython.display import Video, display, Image as IPImage
import remote_infer

## Load the retrained ONNX model

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force ONNX to use CPU for inference

onnx_candidates = list(Path("runs").rglob("face-recognition/train/weights/best.onnx"))
onnx_path = onnx_candidates[0] if onnx_candidates else None
if onnx_path is None:
    print("Local trained ONNX not found. Using pre-trained model from HuggingFace...")
    from huggingface_hub import hf_hub_download
    onnx_path = hf_hub_download(
        repo_id="ariakang/YOLOv11n-face-detection",
        filename="model.onnx"
    )
    print("Note: This is a generic face detector, not your custom-trained model.")
    print("Run notebook 03 first to train a model that recognizes your face.")

model = YOLO(str(onnx_path), task="detect")
print(f"Model loaded from: {onnx_path}")

In [ ]:
def deduplicate_adnan(results, adnan_class_id=0):
    """Keep only the highest-confidence adnan detection per image.
    Any additional adnan detections are reclassified as unknown_face.
    Rationale: one person cannot appear twice in the same photo."""
    for result in results:
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        cls = boxes.cls.cpu().numpy()
        conf = boxes.conf.cpu().numpy()
        adnan_indices = [i for i, c in enumerate(cls) if int(c) == adnan_class_id]
        if len(adnan_indices) > 1:
            # Keep highest confidence, reclassify rest as unknown (class 1)
            best = max(adnan_indices, key=lambda i: conf[i])
            for idx in adnan_indices:
                if idx != best:
                    boxes.cls[idx] = 1  # unknown_face
            print(f"    Dedup: kept 1 adnan ({conf[best]:.2f}), reclassified {len(adnan_indices)-1} as unknown")
    return results

print("Post-processing: deduplicate_adnan() loaded")
print("  Only the highest-confidence adnan detection per image is kept.")
print("  Additional adnan detections are reclassified as unknown_face.")


## Test on still images

In [ ]:
import glob

face_images = sorted(glob.glob("images/test_face_*.jpg"))
print(f"Testing on {len(face_images)} face images\n")

for image_path in face_images:
    results = model.predict(image_path, conf=0.6)
    results = deduplicate_adnan(results)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(8, 5))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

In [ ]:
group_images = sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing on {len(group_images)} group images\n")

for image_path in group_images:
    results = model.predict(image_path, conf=0.6)
    results = deduplicate_adnan(results)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(10, 6))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

## Video Inference — The Demo Highlight

Now for the fun part: we process a video where the model identifies **your face** versus other faces in real-time.

Each frame is run through the ONNX model locally. Your face should appear with a **green** bounding box,
while other faces get a **red** box. The annotated frames are reassembled into an output video.

In [ ]:
video_path = "videos/test_group_video.mov"

if Path(video_path).exists():
    print("Processing video — this may take a minute...")
    output_path = remote_infer.process_video_local(video_path, model, conf=0.6)
    display(Video(output_path, embed=True, width=640))
else:
    print(f"Video not found: {video_path}")
    print("Place a test video (10-30s, you + another person) in the videos/ folder.")

## Deploy trained model to the model server

The model works locally. Now let's upload it to MinIO so the **OpenVINO Model Server** 
(KServe RawDeployment) serves our custom-trained model instead of the generic pre-trained one.

In [ ]:
import boto3
from pathlib import Path
from botocore.config import Config

onnx_candidates = list(Path("runs").rglob("face-recognition/train/weights/best.onnx"))
if not onnx_candidates:
    raise FileNotFoundError("No trained ONNX found. Run notebook 02 first.")
onnx_path = onnx_candidates[0]

BUCKET = "models"
KEY = "face-recognition/1/model.onnx"

s3 = boto3.client("s3",
    endpoint_url="http://minio.minio-storage.svc:9000",
    aws_access_key_id="rhoai-access-key",
    aws_secret_access_key="rhoai-secret-key-12345",
    config=Config(signature_version="s3v4"))

print(f"Uploading {onnx_path} -> s3://{BUCKET}/{KEY}...")
s3.upload_file(str(onnx_path), BUCKET, KEY)

obj = s3.head_object(Bucket=BUCKET, Key=KEY)
size_mb = obj["ContentLength"] / (1024 * 1024)
print(f"Uploaded: s3://{BUCKET}/{KEY} ({size_mb:.1f} MB)")

In [ ]:
import subprocess, time

print("Restarting model server to load the new model...")
result = subprocess.run(
    ["oc", "delete", "pod", "-n", "private-ai",
     "-l", "serving.kserve.io/inferenceservice=face-recognition"],
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

print("Waiting for predictor pod to become Ready...")
for i in range(24):
    time.sleep(5)
    check = subprocess.run(
        ["oc", "get", "inferenceservice", "face-recognition", "-n", "private-ai",
         "-o", "jsonpath={.status.conditions[?(@.type==\"Ready\")].status}"],
        capture_output=True, text=True
    )
    status = check.stdout.strip()
    if status == "True":
        print(f"Model server Ready (took ~{(i+1)*5}s)")
        break
    if i % 3 == 0:
        print(f"  Waiting... ({(i+1)*5}s)")
else:
    print("WARNING: Model server not Ready after 120s. Check: oc get inferenceservice face-recognition -n private-ai")

## Done!

The custom-trained model is now deployed on the model server. 

**Next:** Open **`04-query-model-server.ipynb`** to query it via the production REST API.